In [19]:
import cv2 as cv
import numpy as np
import os
import string
from skimage.metrics import structural_similarity as ssim

In [20]:
mathable_matrix = [
    ['x3', 'o', 'o', 'o', 'o', 'o', 'x3', 'x3', 'o', 'o', 'o', 'o', 'o', 'x3'],
    ['o', 'x2', 'o', 'o', ':', 'o', 'o', 'o', 'o', ':', 'o', 'o', 'x2', 'o'],
    ['o', 'o', 'x2', 'o', 'o', '-', 'o', 'o', '-', 'o', 'o', 'x2', 'o', 'o'],
    ['o', 'o', 'o', 'x2', 'o', 'o', '+', 'x', 'o', 'o', 'x2', 'o', 'o', 'o'],
    ['o', ':', 'o', 'o', 'x2', 'o', 'x', '+', 'o', 'x2', 'o', 'o', ':', 'o'],
    ['o', 'o', '|', 'o', 'o', 'o', 'o', 'o', 'o', 'o', 'o', '|', 'o', 'o'],
    ['x3', 'o', 'o', 'x', '+', 'o', 1, 2, 'o', 'x', '+', 'o', 'o', 'x3'],
    ['x3', 'o', 'o', '+', 'x', 'o', 3, 4, 'o', '+', 'x', 'o', 'o', 'x3'],
    ['o', 'o', '|', 'o', 'o', 'o', 'o', 'o', 'o', 'o', 'o', '|', 'o', 'o'],
    ['o', ':', 'o', 'o', 'x2', 'o', '+', 'x', 'o', 'x2', 'o', 'o', ':', 'o'],
    ['o', 'o', 'o', 'x2', 'o', 'o', 'x', '+', 'o', 'o', 'x2', 'o', 'o', 'o'],
    ['o', 'o', 'x2', 'o', 'o', '-', 'o', 'o', '-', 'o', 'o', 'x2', 'o', 'o'],
    ['o', 'x2', 'o', 'o', ':', 'o', 'o', 'o', 'o', ':', 'o', 'o', 'x2', 'o'],
    ['x3', 'o', 'o', 'o', 'o', 'o', 'x3', 'x3', 'o', 'o', 'o', 'o', 'o', 'x3']
]

## Table processing

In [21]:
def extract_table(image):
    #parametrii descoperiti
    LH = 19
    LS = 149
    LV = 151
    UH = 92
    US = 255
    UV = 255
    img_hsv = cv.cvtColor(image, cv.COLOR_BGR2HSV)
    
    #definirea intervalului pentru masca de culoare
    lower_bound = np.array([LH, LS, LV])
    upper_bound = np.array([UH, US, UV])

    mask = cv.inRange(img_hsv, lower_bound, upper_bound)
    
    left_col = np.argmax(np.any(mask > 0, axis=0))  
    top_row = np.argmax(np.any(mask > 0, axis=1)) 

    right_col = np.argmax(np.any(mask > 0, axis=0)[::-1]) 
    bottom_row = np.argmax(np.any(mask > 0, axis=1)[::-1])

    left_col = left_col
    top_row = top_row
    right_col = image.shape[1] - right_col - 1 
    bottom_row = image.shape[0] - bottom_row - 1 
    
    return left_col, top_row, right_col, bottom_row


In [22]:
def highlight_game_pieces(image):
   
    LH = 9
    LS = 0
    LV = 180
    UH = 90
    US = 71
    UV = 255
    img_hsv = cv.cvtColor(image, cv.COLOR_BGR2HSV)
    
    lower_bound = np.array([LH, LS, LV])
    upper_bound = np.array([UH, US, UV])

    mask = cv.inRange(img_hsv, lower_bound, upper_bound)
    highlighted_image = cv.bitwise_and(image, image, mask=mask)
    return highlighted_image


## Template matching

In [23]:
def piece_matching(square):
    gray_square = cv.cvtColor(square, cv.COLOR_BGR2GRAY)
    
    # determinam daca piesa este cifra sau numar in functie de latimea imaginii
    if gray_square.shape[1] > 48:
        square_resized = cv.resize(gray_square, (70, 55))
        folder = 'templates/numbers'
    else:
        square_resized = cv.resize(gray_square, (37, 55))
        folder = 'templates/digits'

    maxi = -np.inf
    best_template = None
    best_template_image = None

    for template_name in os.listdir(folder):
        template_path = os.path.join(folder, template_name)
        img_template = cv.imread(template_path)

        if img_template is None:
            continue 

        # aplicam resize pe template in functie de criteriul cifra/numar
        if folder == 'templates/numbers':
            img_template_resized = cv.resize(img_template, (70, 55))
        else:
            img_template_resized = cv.resize(img_template, (37, 55))

        img_template_gray = cv.cvtColor(img_template_resized, cv.COLOR_BGR2GRAY)
     
        corr = cv.matchTemplate(square_resized, img_template_gray, cv.TM_CCOEFF_NORMED)
        max_corr = np.max(corr)

        if max_corr > maxi:
            maxi = max_corr
            best_template = template_name
            best_template_image = img_template_gray
            

    return best_template


## Computing the difference matrix

In [24]:
def calculate_diff_matrix(cropped_img1, cropped_img2, square_height, square_width):
    diff_matrix = np.zeros((14, 14))
    for i in range(14):
        for j in range(14):
            y_start = i * square_height
            y_end = (i + 1) * square_height
            x_start = j * square_width
            x_end = (j + 1) * square_width

            square1 = cropped_img1[y_start:y_end, x_start:x_end]
            square2 = cropped_img2[y_start:y_end, x_start:x_end]
            gray1 = cv.cvtColor(square1, cv.COLOR_BGR2GRAY)
            gray2 = cv.cvtColor(square2, cv.COLOR_BGR2GRAY)
            #folosim ssim pentru a detecta diferentele
            score, _ = ssim(gray1, gray2, full=True)
            diff_matrix[i, j] = 1 - score  
    return diff_matrix


In [25]:
def find_max_diff(diff_matrix, threshold, square_height, square_width, cropped_img1):
    max_diff = np.max(diff_matrix)
    if max_diff > threshold:
        max_idx = np.unravel_index(np.argmax(diff_matrix), diff_matrix.shape)
        i, j = max_idx
        y_start, y_end = i * square_height, (i + 1) * square_height
        x_start, x_end = j * square_width, (j + 1) * square_width
        area = cropped_img1[y_start:y_end, x_start:x_end]
        return (i, j, area)
    return None

## Game pieces pre-processing

In [26]:
def crop_coordinates(image, threshold=100, margin=4):

    if len(image.shape) == 3:
        gray_image = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
    else:
        gray_image = image
        
    height, width = gray_image.shape
    top, bottom, left, right = 0, height, 0, width

    for y in range(height):
        if np.any(gray_image[y, :] < threshold):
            top = max(0, y - margin)
            break

    for y in range(height - 1, -1, -1):
        if np.any(gray_image[y, :] < threshold):
            bottom = min(height, y + margin)
            break

    for x in range(width):
        if np.any(gray_image[:, x] < threshold):
            left = max(0, x - margin)
            break

    for x in range(width - 1, -1, -1):
        if np.any(gray_image[:, x] < threshold):
            right = min(width, x + margin)
            break

    return top,bottom,left,right


In [27]:
def binarize_image(image, threshold=127):
 
    if len(image.shape) == 3:
        gray_image = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
    else:
        gray_image = image
    _, binarized_image = cv.threshold(gray_image, threshold, 255, cv.THRESH_BINARY)

    return binarized_image


In [28]:
def crop_margins(image):
    LH =94
    LS = 0
    LV = 0
    UH = 226
    US = 255
    UV = 47
    img_hsv = cv.cvtColor(image, cv.COLOR_BGR2HSV)
    buffer=8
    lower_bound = np.array([LH, LS, LV])
    upper_bound = np.array([UH, US, UV])

    mask = cv.inRange(img_hsv, lower_bound, upper_bound)
    h, w = mask.shape

    #limita de sus
    top = 0
    for i in range(h):
        if 255 in mask[i, :]:
            top = i
            break

    #limita de jos
    bottom = h - 1
    for i in range(h - 1, -1, -1):
        if 255 in mask[i, :]:  
            bottom = i
            break

    #limita din stanga
    left = 0
    for j in range(w):
        if 255 in mask[:, j]: 
            left = j
            break

    #limita din dreapta
    right = w - 1
    for j in range(w - 1, -1, -1):
        if 255 in mask[:, j]:
            right = j
            break

    #extindem limitele cu buffer-ul
    top = max(0, top - buffer)
    bottom = min(h - 1, bottom + buffer)
    left = max(0, left - buffer)
    right = min(w - 1, right + buffer)

    cropped_image = image[top:bottom + 1, left:right + 1]

    return cropped_image

In [29]:
def crop_image(image, top=8, bottom=8, left=8, right=8):
    height, width = image.shape[:2]
    cropped_image = image[top:height-bottom, left:width-right]
    return cropped_image

In [30]:
def template_matching(area):
    #curatam marginile de pixelii mai inchisi la culoare care ar putea impiedica o decupare corecta
    clean_area=crop_image(area)
    
    #taiem marginile ramase
    square_cropped = crop_margins(clean_area)
    binarized = binarize_image(square_cropped, threshold=127)
    
    #taiem cat mai aproape de numar
    top, bottom, left, right = crop_coordinates(binarized)
    cropped_image = square_cropped[top:bottom, left:right]
    
    game_piece_filename = piece_matching(cropped_image)  # Return the filename of the matched template
    return game_piece_filename



## Function that deals with all the tasks logic

In [31]:
def save_results(filename, square, game_piece_filename):
    with open(f"341_Popeanga_Antonia/{filename}.txt", "w") as file:
        piece_number = os.path.splitext(game_piece_filename)[0]
        file.write(f"{square} {piece_number}\n")

In [32]:
def detect_piece_on_board(current_processed,cropped_img1, prev_processed, cropped_img2, filename, scores):
    h, w, _ = cropped_img1.shape
    square_height = h // 14
    square_width = w // 14

    print(f"Processing image {filename}...")
    # matricea diferentelor calculata pe imaginile in care piesele sunt scoase in evidenta
    diff_matrix = calculate_diff_matrix(current_processed, prev_processed, square_height, square_width)

    threshold = 0.3
    #identificam in ce patrat s-a pus piesa
    result = find_max_diff(diff_matrix, threshold, square_height, square_width, cropped_img1)
    
    if result:
        i, j, area = result

        # identificam numarul aflat pe piesa
        game_piece_filename = template_matching(area)
        
        #calculam scorul
        piece_value = int(os.path.splitext(game_piece_filename)[0])
        if mathable_matrix[i][j] == 'x2':
            scores.append(piece_value * 2)
        elif mathable_matrix[i][j] == 'x3':
            scores.append(piece_value * 3)
        else:
            scores.append(piece_value)
        
        #salvam rezultatele
        index_to_letter = {i: letter for i, letter in enumerate(string.ascii_uppercase[:14])}
        square = f"{i+1}{index_to_letter[j]}"
        save_results(filename, square, game_piece_filename)

    return scores


## Generating the scores

In [33]:
def generate_scores_file(turns_file_path, game_number, scores_list):

    with open(turns_file_path, "r") as turns_file:
        turns_lines = turns_file.readlines()

    #generam fisierul de scoruri
    with open(f"341_Popeanga_Antonia/{game_number}_scores.txt", "w") as scores_file:
        for idx, line in enumerate(turns_lines):
            parts = line.strip().split()
            player, start_turn = parts[0], int(parts[1]) - 1
            if idx < len(turns_lines) - 1:
                end_turn = int(turns_lines[idx + 1].split()[1]) - 1
            else:
                end_turn = len(scores_list)

            score_sum = sum(scores_list[start_turn:end_turn])
            scores_file.write(f"{player} {start_turn + 1} {score_sum}\n")


## Main loop of the program

In [ ]:
os.makedirs('341_Popeanga_Antonia', exist_ok=True)
empty_table = cv.imread("templates/01.jpg")
prev_image = empty_table

#aici trebuie introdus path-ul catre folderul cu poze
test_file='path/catre/folderul/test'

left_col, top_row, right_col, bottom_row = extract_table(empty_table)
files = sorted(os.listdir(test_file))
game_number=1
scores = []


for file in files:
    if file.endswith("_turns.txt"):
        #generam fisierul de scoruri pentru fiecare joc
        generate_scores_file(os.path.join(test_file, file), game_number, scores)
        
        #resetam param pentru un nou joc
        scores = []
        game_number += 1
        prev_image = empty_table

    elif file.endswith('.jpg'):
        current_img = cv.imread(os.path.join(test_file, file))

        #decuparea matricei din tabla
        current_cropped = current_img[top_row:bottom_row, left_col:right_col]
        prev_cropped = prev_image[top_row:bottom_row, left_col:right_col]

        #scoatem in evidenta piesele jocului
        current_processed=highlight_game_pieces(current_cropped)
        prev_processed=highlight_game_pieces(prev_cropped)
        
        #detectarea pozitiei, piesei si calcularea scorului
        scores = detect_piece_on_board(current_processed,current_cropped, prev_processed, prev_cropped,os.path.splitext(file)[0], scores)
        prev_image = current_img
print("Done!")

Processing image 1_01...
Processing image 1_02...
Processing image 1_03...
Processing image 1_04...
Processing image 1_05...
Processing image 1_06...
Processing image 1_07...
Processing image 1_08...
Processing image 1_09...
Processing image 1_10...
Processing image 1_11...
Processing image 1_12...
Processing image 1_13...
Processing image 1_14...
Processing image 1_15...
Processing image 1_16...
Processing image 1_17...
Processing image 1_18...
Processing image 1_19...
Processing image 1_20...
Processing image 1_21...
Processing image 1_22...
Processing image 1_23...
Processing image 1_24...
Processing image 1_25...
Processing image 1_26...
Processing image 1_27...
Processing image 1_28...
Processing image 1_29...
Processing image 1_30...
Processing image 1_31...
Processing image 1_32...
Processing image 1_33...
Processing image 1_34...
Processing image 1_35...
Processing image 1_36...
Processing image 1_37...
Processing image 1_38...
Processing image 1_39...
Processing image 1_40...
